# Notebook 35 — Exact coverage, market regimes and paired trading signals

**Research question:** Does chronological OOF stacking select higher-quality Up trades than full-history RF when all models are forced to select the **same number of validation observations**?

**Inputs:** Notebook 34's saved *development-fold* validation predictions. No retraining or historical final-holdout access. The retrospective exact top-k rule uses the entire validation score distribution; **this is a ranking diagnostic, not a live trading policy**. Trading results are indicative, not realized P&L. We use raw probabilities; Platt calibration in Notebook 34 preserved rankings.

In [ ]:
from pathlib import Path
import sys
ROOT=Path.cwd().resolve()
if ROOT.name=='notebooks': ROOT=ROOT.parent
if not (ROOT/'src').exists(): raise FileNotFoundError('Open this notebook inside stock_ml_dynamic')
sys.path.insert(0,str(ROOT/'src'))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
from stock_ml_dynamic.exact_coverage_regime_research import run_research, load_predictions
INPUT=ROOT/'artifacts/notebook34_calibration_coverage/validation_predictions.parquet'
OUTPUT=ROOT/'artifacts/notebook35_exact_coverage_regimes'
COVERAGES=(.05,.10,.15,.20,.25)
BOOTSTRAP_REPS=1000
SEED=42
print('Input:',INPUT,'exists:',INPUT.exists())

## 1. Load saved out-of-sample predictions from Notebook 34

No models are fitted in Notebook 35. If this file is absent, run Notebook 34 and ensure its outputs were written to the repository-root `artifacts/` directory.

In [ ]:
preds=load_predictions(INPUT)
print('Rows:',len(preds),'Folds:',sorted(preds.fold.unique()),'Models:',sorted(preds.model.unique()))
print('Date range:',preds.bar_timestamp.min(),preds.bar_timestamp.max())
print('Unique symbols:',preds.symbol.nunique())
display(preds.groupby(['fold','model','calibration']).size().unstack(['model','calibration']))

## 2. Exact top-k and paired selection comparisons

For each outer fold, every model selects exactly `round(coverage × number of validation rows)` observations. Rankings are computed retrospectively on the validation fold. The same labels and returns are used for each model, and all model prediction keys must align.

In [ ]:
result=run_research(INPUT,output_dir=OUTPUT,coverages=COVERAGES,reps=BOOTSTRAP_REPS,seed=SEED)
scores=result['exact_scores']; paired=result['paired']; boot=result['bootstrap']; regimes=result['regimes']; regime_scores=result['regime_scores']
assert scores.groupby(['fold','target_coverage']).k.nunique().eq(1).all()
assert scores.groupby(['fold','target_coverage']).entries.nunique().eq(1).all()
print('Policy rows:',len(scores),'Paired selection rows:',len(paired),'Bootstrap rows:',len(boot))
display(scores.groupby(['target_coverage','model'])[['up_precision','mean_net','actual_coverage']].mean().round(5))

## 3. Paired fold-by-fold differences vs full-history RF

In [ ]:
baseline=scores.loc[scores.model.eq('full_history_rf')].set_index(['fold','target_coverage'])
comparisons=[]
for model in ['oof_meta_logistic','oof_meta_rf']:
    for _,row in scores[scores.model.eq(model)].iterrows():
        b=baseline.loc[(row.fold,row.target_coverage)]
        comparisons.append({'fold':row.fold,'coverage':row.target_coverage,'model':model,
                            'delta_up_precision_pp':100*(row.up_precision-b.up_precision),
                            'delta_net_bps':10000*(row.mean_net-b.mean_net)})
comparisons=pd.DataFrame(comparisons)
display(comparisons.round(4))
display(comparisons.groupby(['coverage','model'])[['delta_up_precision_pp','delta_net_bps']].mean().round(4))

## 4. Shared vs unique trades

The `shared` bucket contains trades selected by both models. `baseline_only` and `challenger_only` have equal counts by construction; differences in their average returns directly explain the ranking advantage or disadvantage.

In [ ]:
display(paired.groupby(['target_coverage','challenger','selection'])[['entries','up_precision','mean_net']].mean().round(5))
subset=paired.loc[paired.target_coverage.eq(.20)]
fig,ax=plt.subplots(figsize=(9,4))
for model,g in subset.groupby('challenger'):
    values=g.groupby('selection').mean_net.mean()
    ax.plot(values.index,values.values*10000,marker='o',label=model)
ax.set_ylabel('Mean net return (basis points)');ax.set_title('20% coverage: shared vs unique selections')
ax.legend();plt.tight_layout();plt.show()

## 5. Historical market-regime diagnostics

Regime proxy: cross-sectional mean next-bar return, lagged **two dates** before its use. Bull/bear is the sign of the trailing 20-day mean; high/low volatility compares the trailing 20-day volatility with a prior-only 60-day rolling median. This is a *validation-period descriptive proxy*, not a trained regime classifier. Warm-up dates with insufficient history are excluded.

In [ ]:
print('Regime-eligible dates:',regimes.dropna(subset=['trend_regime','vol_regime']).shape[0], 'of',len(regimes))
display(regime_scores.groupby(['target_coverage','model','regime_type','regime'])[['entries','up_precision','mean_net']].mean().round(5))
for coverage in [.10,.20]:
    sub=regime_scores[(regime_scores.target_coverage==coverage)&(regime_scores.regime_type=='trend_regime')]
    pivot=sub.groupby(['model','regime']).mean_net.mean().unstack()
    display(pivot.rename_axis(index='Model').round(5))

## 6. Date-block bootstrap confidence intervals

Resample entire trading dates **within each outer fold**, retaining all stocks and paired model predictions on each sampled date. The intervals describe uncertainty conditional on this historical design; they do not establish a deployable edge. Multiple coverage/model comparisons are exploratory.

In [ ]:
view=boot.copy()
for field in ['point_estimate','ci_low','ci_high']:
    view[field]=view[field]*100
print('Precision differences in percentage points; return differences in percentage points')
display(view[view.target_coverage.isin([.10,.20,.25])].round(4))
print('Intervals excluding zero by metric:')
display(boot.assign(excludes_zero=(boot.ci_low>0)|(boot.ci_high<0)).groupby(['challenger','metric']).excludes_zero.agg(['sum','count']))

## 7. Interpret and decide

- Prefer comparisons with the same **exact k** in each fold; check the paired confidence intervals and differences across folds.
- Compare unique trades, not just shared trades.
- Look for improvements that persist across market regimes and coverage settings.
- **Do not** use validation-wide top-k selection as a deployable strategy. Live thresholds must be learned from earlier data and applied prospectively.
- Historical final holdout remains closed. Prospective ledger is untouched.

In [ ]:
print('Saved research tables to:',OUTPUT)
print('No training performed. Closed final historical holdout and prospective ledger were not accessed.')